# 🚀 Chat Data Export Orchestrator v2.1

## Professional Multi-Platform Chat Export Tool

**Features:**
- 🎯 **Multi-Platform Support**: ChatGPT, Claude, Gemini, and auto-detection
- 💾 **Memory-Efficient**: Stream processing for files of any size
- 🎨 **Interactive UI**: Beautiful ipywidgets-based interface
- 🧹 **User Message Cleaning**: Length gate, exact/near dedup with presets
- 📚 **Monolith Export**: Merged files with provenance tracking
- 📊 **Analytics Dashboard**: Real-time statistics and visualizations
- ⚡ **Optimized for Colab Pro**: GPU acceleration and parallel processing

---

### 📖 Quick Start Guide

1. **Run Setup** → Install dependencies (Cell below)
2. **Configure** → Use the interactive UI to set preferences
3. **Upload Files** → Drop your JSON files or use the upload widget
4. **Process** → Click Process button and watch the magic!
5. **Analyze** → View statistics and download results

### 🆕 New in v2.1
- **User Message Cleaning**: Filter short, duplicate, and near-duplicate user messages
- **Cleaning Presets**: Quick settings (none/light/heavy) for common use cases
- **Monolith Export**: Create single merged files with full provenance
- **Source File Tracking**: Know which JSON files each conversation came from

In [1]:
# Install Dependencies
!pip install -q ijson memory-profiler ipywidgets plotly wordcloud pandas numpy tqdm chardet pyarrow jinja2
!jupyter nbextension enable --py widgetsnbextension --sys-prefix 2>/dev/null || true

print("✅ Dependencies installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.3/148.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:00
✅ Dependencies installed successfully!


In [2]:
# Import all required libraries
import json
import os
import re
import hashlib
import gzip
import zipfile
import shutil
import gc
from pathlib import Path
from typing import Dict, List, Optional, Union, Tuple, Any, Generator
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache, wraps
from abc import ABC, abstractmethod
import traceback
import logging
from enum import Enum

import ijson
from memory_profiler import profile
import ipywidgets as widgets
from IPython.display import display, HTML, Javascript, clear_output
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from wordcloud import WordCloud
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import chardet
from jinja2 import Template

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('ChatExportOrchestrator')

print("✨ Environment ready!")

✨ Environment ready!


In [3]:
# Core Data Structures and Enums

class Platform(Enum):
    CHATGPT = "chatgpt"
    CLAUDE = "claude"
    GEMINI = "gemini"
    UNKNOWN = "unknown"

class ExportFormat(Enum):
    MARKDOWN = "md"
    TEXT = "txt"
    JSON = "json"
    HTML = "html"

class ProcessingMode(Enum):
    MEMORY = "memory"
    STREAM = "stream"
    CHUNK = "chunk"

@dataclass
class Message:
    """Represents a single message in a conversation."""
    role: str
    content: str
    timestamp: Optional[str] = None
    metadata: Dict = field(default_factory=dict)

    def __hash__(self):
        return hash((self.role, self.content[:100] if self.content else ''))

@dataclass
class Conversation:
    """Represents a complete conversation."""
    id: str
    title: str
    messages: List[Message]
    created_at: Optional[str] = None
    updated_at: Optional[str] = None
    platform: Platform = Platform.UNKNOWN
    metadata: Dict = field(default_factory=dict)

    @property
    def message_count(self) -> int:
        return len(self.messages)

    @property
    def word_count(self) -> int:
        return sum(len(msg.content.split()) for msg in self.messages if msg.content)

    @property
    def unique_hash(self) -> str:
        content = ''.join(msg.content[:50] for msg in self.messages[:3])
        return hashlib.md5(content.encode()).hexdigest()

@dataclass
class ProcessingStats:
    """Statistics for processing operations."""
    total_files: int = 0
    processed_files: int = 0
    total_conversations: int = 0
    exported_conversations: int = 0
    total_messages: int = 0
    errors: List[str] = field(default_factory=list)
    processing_time: float = 0.0
    memory_peak: float = 0.0
    platform_breakdown: Dict[Platform, int] = field(default_factory=dict)

    def add_conversation(self, conv: Conversation):
        self.total_conversations += 1
        self.total_messages += conv.message_count
        if conv.platform not in self.platform_breakdown:
            self.platform_breakdown[conv.platform] = 0
        self.platform_breakdown[conv.platform] += 1

@dataclass
class ExportConfig:
    """Configuration for export operations."""
    input_path: Path
    output_path: Path
    subset: str = 'both'
    format: ExportFormat = ExportFormat.MARKDOWN
    export_code: bool = True
    keywords: List[str] = field(default_factory=list)
    group_by_keywords: bool = True
    unique_groups: bool = False
    max_filename_len: int = 160
    processing_mode: ProcessingMode = ProcessingMode.STREAM
    chunk_size: int = 1000
    parallel_workers: int = 4
    date_filter_start: Optional[datetime] = None
    date_filter_end: Optional[datetime] = None
    min_message_count: int = 0
    max_message_count: Optional[int] = None
    deduplicate: bool = True
    verbose: bool = True
    dry_run: bool = False

    def validate(self) -> List[str]:
        errors = []
        if not self.input_path.exists():
            errors.append(f"Input path does not exist: {self.input_path}")
        if self.max_filename_len < 50:
            errors.append("Maximum filename length must be at least 50")
        if self.chunk_size < 100:
            errors.append("Chunk size must be at least 100")
        if self.parallel_workers < 1:
            errors.append("Must have at least 1 parallel worker")
        return errors

print("✅ Core classes defined")

✅ Core classes defined


In [4]:
# Platform Parsers

class BaseChatParser(ABC):
    """Abstract base class for chat parsers."""

    @abstractmethod
    def parse(self, data: Union[Dict, List]) -> List[Conversation]:
        pass

    @abstractmethod
    def detect_format(self, data: Union[Dict, List]) -> bool:
        pass

    def _safe_get(self, data: Dict, path: str, default=None):
        keys = path.split('.')
        value = data
        for key in keys:
            if isinstance(value, dict):
                value = value.get(key, default)
            else:
                return default
        return value if value is not None else default

    def _parse_timestamp(self, timestamp: Any) -> Optional[str]:
        if not timestamp:
            return None
        if isinstance(timestamp, (int, float)):
            try:
                dt = datetime.fromtimestamp(timestamp)
                return dt.isoformat()
            except:
                pass
        return str(timestamp)

class ChatGPTParser(BaseChatParser):
    def detect_format(self, data: Union[Dict, List]) -> bool:
        sample = data[0] if isinstance(data, list) and data else data
        return isinstance(sample, dict) and 'mapping' in sample

    def parse(self, data: Union[Dict, List]) -> List[Conversation]:
        conversations = []
        if isinstance(data, dict):
            data = [data]

        for conv_data in data:
            try:
                conv_id = conv_data.get('id', hashlib.md5(str(conv_data).encode()).hexdigest()[:8])
                title = conv_data.get('title', 'Untitled')
                created_at = self._parse_timestamp(conv_data.get('create_time'))
                updated_at = self._parse_timestamp(conv_data.get('update_time'))

                messages = []
                mapping = conv_data.get('mapping', {})

                for node_id, node_data in mapping.items():
                    message = node_data.get('message')
                    if message and message.get('content'):
                        content_parts = message['content'].get('parts', [])
                        if content_parts:
                            content = '\n'.join(str(part) for part in content_parts)
                            author = self._safe_get(message, 'author.role', 'unknown')
                            if author == 'system':
                                continue
                            role = 'user' if author == 'user' else 'assistant'
                            msg = Message(
                                role=role,
                                content=content,
                                timestamp=self._parse_timestamp(message.get('create_time')),
                                metadata={'id': message.get('id')}
                            )
                            messages.append(msg)

                if messages:
                    conversations.append(Conversation(
                        id=conv_id,
                        title=title,
                        messages=messages,
                        created_at=created_at,
                        updated_at=updated_at,
                        platform=Platform.CHATGPT
                    ))
            except Exception as e:
                logger.warning(f"Error parsing ChatGPT conversation: {e}")
        return conversations

class ClaudeParser(BaseChatParser):
    def detect_format(self, data: Union[Dict, List]) -> bool:
        sample = data[0] if isinstance(data, list) and data else data
        return isinstance(sample, dict) and (
            'chat_messages' in sample or 'uuid' in sample or
            (sample.get('messages') and any('sender' in msg for msg in sample.get('messages', []) if isinstance(msg, dict)))
        )

    def parse(self, data: Union[Dict, List]) -> List[Conversation]:
        conversations = []
        if isinstance(data, dict):
            data = [data]

        for conv_data in data:
            try:
                conv_id = conv_data.get('id', conv_data.get('uuid', hashlib.md5(str(conv_data).encode()).hexdigest()[:8]))
                title = conv_data.get('title', conv_data.get('name', 'Untitled'))
                messages = []
                message_list = conv_data.get('messages') or conv_data.get('chat_messages') or conv_data.get('conversation', [])

                for msg_data in message_list:
                    role = msg_data.get('role', msg_data.get('sender', ''))
                    if role in ['human', 'user']:
                        role = 'user'
                    elif role in ['assistant', 'claude', 'ai']:
                        role = 'assistant'
                    else:
                        continue

                    content = msg_data.get('content', msg_data.get('text', msg_data.get('message', '')))
                    if isinstance(content, list):
                        content = '\n'.join(str(item) for item in content)
                    elif isinstance(content, dict):
                        content = content.get('text', str(content))

                    if content:
                        messages.append(Message(
                            role=role,
                            content=str(content),
                            timestamp=self._parse_timestamp(msg_data.get('timestamp', msg_data.get('created_at'))),
                            metadata=msg_data.get('metadata', {})
                        ))

                if messages:
                    conversations.append(Conversation(
                        id=conv_id,
                        title=title,
                        messages=messages,
                        created_at=self._parse_timestamp(conv_data.get('created_at')),
                        updated_at=self._parse_timestamp(conv_data.get('updated_at')),
                        platform=Platform.CLAUDE
                    ))
            except Exception as e:
                logger.warning(f"Error parsing Claude conversation: {e}")
        return conversations

class GeminiParser(BaseChatParser):
    def detect_format(self, data: Union[Dict, List]) -> bool:
        sample = data[0] if isinstance(data, list) and data else data
        if not isinstance(sample, dict):
            return False
        return (
            'entries' in sample or
            (sample.get('messages') and any('model' in msg or 'type' in msg for msg in sample.get('messages', []) if isinstance(msg, dict)))
        )

    def parse(self, data: Union[Dict, List]) -> List[Conversation]:
        conversations = []
        if isinstance(data, dict):
            if 'conversations' in data:
                data = data['conversations']
            else:
                data = [data]

        for conv_data in data:
            try:
                conv_id = conv_data.get('id', hashlib.md5(str(conv_data).encode()).hexdigest()[:8])
                title = conv_data.get('title', 'Untitled')
                messages = []
                message_list = conv_data.get('messages') or conv_data.get('entries') or conv_data.get('chats', [])

                for msg_data in message_list:
                    role = msg_data.get('role', msg_data.get('type', ''))
                    if role in ['user', 'prompt']:
                        role = 'user'
                    elif role in ['model', 'gemini', 'response', 'assistant']:
                        role = 'assistant'
                    else:
                        continue

                    content = msg_data.get('content', msg_data.get('text', msg_data.get('message', '')))
                    if isinstance(content, dict):
                        content = content.get('text', str(content))
                    elif isinstance(content, list):
                        content = '\n'.join(str(item) for item in content)

                    if content:
                        messages.append(Message(
                            role=role,
                            content=str(content),
                            timestamp=self._parse_timestamp(msg_data.get('timestamp')),
                            metadata=msg_data.get('metadata', {})
                        ))

                if messages:
                    conversations.append(Conversation(
                        id=conv_id,
                        title=title,
                        messages=messages,
                        created_at=self._parse_timestamp(conv_data.get('created_at')),
                        updated_at=self._parse_timestamp(conv_data.get('updated_at')),
                        platform=Platform.GEMINI
                    ))
            except Exception as e:
                logger.warning(f"Error parsing Gemini conversation: {e}")
        return conversations

class UniversalChatParser:
    def __init__(self):
        self.parsers = [ChatGPTParser(), ClaudeParser(), GeminiParser()]

    def detect_platform(self, data: Union[Dict, List]) -> Platform:
        for parser in self.parsers:
            if parser.detect_format(data):
                if isinstance(parser, ChatGPTParser):
                    return Platform.CHATGPT
                elif isinstance(parser, ClaudeParser):
                    return Platform.CLAUDE
                elif isinstance(parser, GeminiParser):
                    return Platform.GEMINI
        return Platform.UNKNOWN

    def parse(self, data: Union[Dict, List], platform: Optional[Platform] = None) -> List[Conversation]:
        if platform is None:
            platform = self.detect_platform(data)

        parser_map = {
            Platform.CHATGPT: ChatGPTParser(),
            Platform.CLAUDE: ClaudeParser(),
            Platform.GEMINI: GeminiParser()
        }

        parser = parser_map.get(platform)
        if parser:
            return parser.parse(data)

        for parser in self.parsers:
            try:
                conversations = parser.parse(data)
                if conversations:
                    return conversations
            except:
                continue
        return []

print("✅ Parser classes defined")

✅ Parser classes defined


In [5]:
# File Processors and Filters

class FileProcessor:
    @staticmethod
    def detect_encoding(file_path: Path) -> str:
        with open(file_path, 'rb') as f:
            raw = f.read(4)

        if raw.startswith(b'\\xff\\xfe\\x00\\x00'):
            return 'utf-32-le'
        elif raw.startswith(b'\\x00\\x00\\xfe\\xff'):
            return 'utf-32-be'
        elif raw.startswith(b'\\xff\\xfe'):
            return 'utf-16-le'
        elif raw.startswith(b'\\xfe\\xff'):
            return 'utf-16-be'
        elif raw.startswith(b'\\xef\\xbb\\xbf'):
            return 'utf-8-sig'
        elif raw.startswith(b'\\x1f\\x8b'):
            return 'gzip'
        elif raw.startswith(b'PK'):
            return 'zip'

        with open(file_path, 'rb') as f:
            result = chardet.detect(f.read(10000))
            if result['confidence'] > 0.7:
                return result['encoding']
        return 'utf-8'

    @staticmethod
    def load_json_file(filepath: Path, mode: ProcessingMode = ProcessingMode.STREAM) -> Any:
        encoding = FileProcessor.detect_encoding(filepath)

        if encoding == 'gzip':
            with gzip.open(filepath, 'rt', encoding='utf-8') as f:
                return json.load(f)
        elif encoding == 'zip':
            with zipfile.ZipFile(filepath, 'r') as zf:
                for name in zf.namelist():
                    if name.endswith('.json'):
                        with zf.open(name) as f:
                            return json.loads(f.read().decode('utf-8'))

        if mode == ProcessingMode.MEMORY:
            with open(filepath, 'r', encoding=encoding) as f:
                return json.load(f)
        elif mode == ProcessingMode.STREAM:
            return FileProcessor._stream_json(filepath, encoding)
        else:
            with open(filepath, 'r', encoding=encoding) as f:
                return json.load(f)

    @staticmethod
    def _stream_json(filepath: Path, encoding: str) -> Generator:
        with open(filepath, 'rb') as f:
            try:
                parser = ijson.items(f, 'item')
                for item in parser:
                    yield item
            except:
                f.seek(0)
                data = json.load(f)
                if isinstance(data, list):
                    for item in data:
                        yield item
                else:
                    yield data

class StreamProcessor:
    def __init__(self, parser: UniversalChatParser):
        self.parser = parser
        self.stats = ProcessingStats()
        self.source_files = {}  # Track source files for conversations

    def process_file(self, filepath: Path, config: ExportConfig) -> List[Conversation]:
        conversations = []
        try:
            file_size_mb = filepath.stat().st_size / (1024**2)
            logger.info(f"Processing {filepath.name} ({file_size_mb:.1f} MB)")

            if file_size_mb > 100 and config.processing_mode == ProcessingMode.STREAM:
                for item in FileProcessor.load_json_file(filepath, ProcessingMode.STREAM):
                    parsed = self.parser.parse(item)
                    conversations.extend(parsed)
                    if len(conversations) % 100 == 0:
                        gc.collect()
            else:
                data = FileProcessor.load_json_file(filepath, ProcessingMode.MEMORY)
                conversations = self.parser.parse(data)

            # Track source file for each conversation
            for conv in conversations:
                if conv.id not in self.source_files:
                    self.source_files[conv.id] = []
                self.source_files[conv.id].append(filepath.name)
                self.stats.add_conversation(conv)

            return conversations
        except Exception as e:
            error_msg = f"Error processing {filepath.name}: {str(e)}"
            logger.error(error_msg)
            self.stats.errors.append(error_msg)
            return []

    def process_directory(self, directory: Path, config: ExportConfig) -> List[Conversation]:
        all_conversations = []
        json_files = list(directory.glob('**/*.json')) + list(directory.glob('**/*.jsonl'))
        self.stats.total_files = len(json_files)

        if config.parallel_workers > 1 and len(json_files) > 1:
            with ThreadPoolExecutor(max_workers=config.parallel_workers) as executor:
                futures = {executor.submit(self.process_file, f, config): f for f in json_files}
                for future in tqdm(as_completed(futures), total=len(json_files), desc="Processing files"):
                    conversations = future.result()
                    all_conversations.extend(conversations)
                    self.stats.processed_files += 1
        else:
            for json_file in tqdm(json_files, desc="Processing files"):
                conversations = self.process_file(json_file, config)
                all_conversations.extend(conversations)
                self.stats.processed_files += 1

        return all_conversations

class ConversationFilter:
    @staticmethod
    def apply_filters(conversations: List[Conversation], config: ExportConfig) -> List[Conversation]:
        filtered = conversations

        if config.date_filter_start or config.date_filter_end:
            filtered = ConversationFilter._filter_by_date(filtered, config.date_filter_start, config.date_filter_end)

        if config.min_message_count > 0 or config.max_message_count:
            filtered = ConversationFilter._filter_by_message_count(filtered, config.min_message_count, config.max_message_count)

        if config.keywords:
            filtered = ConversationFilter._filter_by_keywords(filtered, config.keywords)

        if config.deduplicate:
            filtered = ConversationFilter._deduplicate(filtered)

        return filtered

    @staticmethod
    def _filter_by_date(conversations: List[Conversation], start: Optional[datetime], end: Optional[datetime]) -> List[Conversation]:
        filtered = []
        for conv in conversations:
            if conv.created_at:
                try:
                    conv_date = datetime.fromisoformat(conv.created_at.replace('Z', '+00:00'))
                    if start and conv_date < start:
                        continue
                    if end and conv_date > end:
                        continue
                    filtered.append(conv)
                except:
                    filtered.append(conv)
            else:
                filtered.append(conv)
        return filtered

    @staticmethod
    def _filter_by_message_count(conversations: List[Conversation], min_count: int, max_count: Optional[int]) -> List[Conversation]:
        filtered = []
        for conv in conversations:
            if conv.message_count < min_count:
                continue
            if max_count and conv.message_count > max_count:
                continue
            filtered.append(conv)
        return filtered

    @staticmethod
    def _filter_by_keywords(conversations: List[Conversation], keywords: List[str]) -> List[Conversation]:
        filtered = []
        keywords_lower = [kw.lower() for kw in keywords]
        for conv in conversations:
            full_text = conv.title.lower()
            for msg in conv.messages:
                full_text += " " + msg.content.lower()
            if any(kw in full_text for kw in keywords_lower):
                filtered.append(conv)
        return filtered

    @staticmethod
    def _deduplicate(conversations: List[Conversation]) -> List[Conversation]:
        seen_hashes = set()
        unique_conversations = []
        for conv in conversations:
            conv_hash = conv.unique_hash
            if conv_hash not in seen_hashes:
                seen_hashes.add(conv_hash)
                unique_conversations.append(conv)
        logger.info(f"Deduplication: {len(conversations)} -> {len(unique_conversations)} conversations")
        return unique_conversations

class UserMessageFilter:
    """Filter user messages based on length, exact duplicates, and near duplicates."""

    def __init__(self, config: ExportConfig):
        self.config = config
        self.stats = {
            'kept_user': 0,
            'dropped_short': 0,
            'dropped_exact': 0,
            'dropped_neardup': 0
        }
        self.global_exact_hashes = set()

    @staticmethod
    def normalize_text(text: str) -> str:
        """Normalize text for filtering decisions only."""
        # Lowercase
        normalized = text.lower()

        # Strip triple backticks
        normalized = re.sub(r'```[^`]*```', '', normalized)

        # Strip inline backticks
        normalized = re.sub(r'`[^`]+`', '', normalized)

        # Replace URLs with <URL>
        normalized = re.sub(r'https?://[^\\s]+', '<URL>', normalized)

        # Collapse whitespace
        normalized = ' '.join(normalized.split())

        return normalized

    @staticmethod
    def jaccard_similarity(text1: str, text2: str) -> float:
        """Calculate Jaccard similarity between two texts based on word tokens."""
        words1 = set(text1.split())
        words2 = set(text2.split())

        if not words1 or not words2:
            return 0.0

        intersection = words1.intersection(words2)
        union = words1.union(words2)

        return len(intersection) / len(union) if union else 0.0

    def filter_conversation(self, conversation: Conversation) -> Conversation:
        """Filter user messages in a conversation."""
        if not self.config.user_cleaning_enabled:
            return conversation

        filtered_messages = []
        conversation_exact_hashes = set()
        recent_user_normalized = []  # For near-dup checking

        for msg in conversation.messages:
            # Only filter user messages
            if msg.role != 'user':
                filtered_messages.append(msg)
                continue

            # Normalize for filtering decisions
            normalized = self.normalize_text(msg.content)
            keep = True

            # Length gate
            if self.config.length_gate_enabled and len(normalized) < self.config.min_user_chars:
                keep = False
                self.stats['dropped_short'] += 1

            # Exact dedup
            if keep and self.config.exact_dedup_enabled:
                text_hash = hashlib.md5(normalized.encode()).hexdigest()

                if self.config.dedup_scope == 'conversation':
                    if text_hash in conversation_exact_hashes:
                        keep = False
                        self.stats['dropped_exact'] += 1
                    else:
                        conversation_exact_hashes.add(text_hash)
                else:  # global
                    if text_hash in self.global_exact_hashes:
                        keep = False
                        self.stats['dropped_exact'] += 1
                    else:
                        self.global_exact_hashes.add(text_hash)

            # Near-dup
            if keep and self.config.near_dup_enabled:
                for prev_normalized in recent_user_normalized[-self.config.near_dup_window:]:
                    similarity = self.jaccard_similarity(normalized, prev_normalized)
                    if similarity >= self.config.near_dup_threshold:
                        keep = False
                        self.stats['dropped_neardup'] += 1
                        break

            if keep:
                filtered_messages.append(msg)
                recent_user_normalized.append(normalized)
                self.stats['kept_user'] += 1

        # Return conversation with filtered messages
        return Conversation(
            id=conversation.id,
            title=conversation.title,
            messages=filtered_messages,
            created_at=conversation.created_at,
            updated_at=conversation.updated_at,
            platform=conversation.platform,
            metadata=conversation.metadata
        )

    def filter_conversations(self, conversations: List[Conversation]) -> List[Conversation]:
        """Filter user messages across all conversations."""
        filtered_conversations = []

        for conv in conversations:
            filtered_conv = self.filter_conversation(conv)
            # Only keep conversations that still have messages
            if filtered_conv.messages:
                filtered_conversations.append(filtered_conv)

        return filtered_conversations

    def print_stats(self):
        """Print filtering statistics."""
        if self.config.report_user_filter_stats:
            print("\\n📊 User Message Filtering Statistics:")
            print(f"  ✅ Kept user messages: {self.stats['kept_user']}")
            print(f"  ❌ Dropped (too short): {self.stats['dropped_short']}")
            print(f"  ❌ Dropped (exact dup): {self.stats['dropped_exact']}")
            print(f"  ❌ Dropped (near dup): {self.stats['dropped_neardup']}")
            total_dropped = sum([self.stats['dropped_short'], self.stats['dropped_exact'], self.stats['dropped_neardup']])
            total_processed = self.stats['kept_user'] + total_dropped
            if total_processed > 0:
                print(f"  📈 Filter rate: {(total_dropped / total_processed * 100):.1f}%")

print("✅ Processor classes defined")

✅ Processor classes defined


In [6]:
# Export System

class CodeExtractor:
    LANGUAGE_EXTENSIONS = {
        'python': '.py', 'javascript': '.js', 'typescript': '.ts',
        'java': '.java', 'cpp': '.cpp', 'c': '.c', 'csharp': '.cs',
        'html': '.html', 'css': '.css', 'sql': '.sql', 'bash': '.sh',
        'shell': '.sh', 'yaml': '.yml', 'json': '.json', 'xml': '.xml',
        'markdown': '.md', 'rust': '.rs', 'go': '.go', 'ruby': '.rb',
        'php': '.php', 'swift': '.swift', 'kotlin': '.kt', 'r': '.r'
    }

    @staticmethod
    def extract_code_blocks(content: str) -> List[Tuple[str, str]]:
        code_blocks = []
        pattern = r'```(\\w+)?\\n?(.*?)```'
        matches = re.findall(pattern, content, re.DOTALL)

        for language, code in matches:
            language = language.lower() if language else 'text'
            code_blocks.append((code.strip(), language))

        if not code_blocks:
            inline_pattern = r'`([^`]+)`'
            inline_matches = re.findall(inline_pattern, content)
            for code in inline_matches:
                if len(code) > 50:
                    code_blocks.append((code, 'text'))

        return code_blocks

class ExportFormatter:
    @staticmethod
    def format_markdown(conversation: Conversation, subset: str) -> str:
        lines = [f"# {conversation.title}\\n"]
        lines.append(f"**Platform:** {conversation.platform.value}\\n")
        if conversation.created_at:
            lines.append(f"**Date:** {conversation.created_at}\\n")
        lines.append(f"**Messages:** {conversation.message_count}\\n")
        lines.append(f"**Words:** {conversation.word_count}\\n")
        lines.append("\\n---\\n\\n")

        for msg in conversation.messages:
            if subset == 'both' or subset == msg.role:
                role_emoji = "👤" if msg.role == 'user' else "🤖"
                lines.append(f"### {role_emoji} {msg.role.title()}\\n\\n")
                lines.append(f"{msg.content}\\n\\n")
                if msg.timestamp:
                    lines.append(f"*{msg.timestamp}*\\n\\n")
                lines.append("---\\n\\n")

        return ''.join(lines)

    @staticmethod
    def format_text(conversation: Conversation, subset: str) -> str:
        lines = [f"{conversation.title}\\n"]
        lines.append(f"Platform: {conversation.platform.value}\\n")
        lines.append("=" * 60 + "\\n\\n")

        for msg in conversation.messages:
            if subset == 'both' or subset == msg.role:
                lines.append(f"[{msg.role.upper()}]:\\n")
                lines.append(f"{msg.content}\\n\\n")
                lines.append("-" * 40 + "\\n\\n")

        return ''.join(lines)

    @staticmethod
    def format_json(conversation: Conversation, subset: str) -> str:
        messages = []
        for msg in conversation.messages:
            if subset == 'both' or subset == msg.role:
                messages.append({
                    'role': msg.role,
                    'content': msg.content,
                    'timestamp': msg.timestamp
                })

        export_data = {
            'id': conversation.id,
            'title': conversation.title,
            'platform': conversation.platform.value,
            'created_at': conversation.created_at,
            'updated_at': conversation.updated_at,
            'message_count': len(messages),
            'messages': messages
        }
        return json.dumps(export_data, indent=2, ensure_ascii=False)

class ChatExporter:
    def __init__(self, config: ExportConfig):
        self.config = config
        self.stats = ProcessingStats()
        self.exported_files = []
        self.filename_cache = {}
        self.conversation_export_map = {}  # Map conv ID to export file

        self.default_dir = config.output_path / "conversations"
        self.code_dir = config.output_path / "code_blocks"
        self.keyword_dirs = {}

        if not config.dry_run:
            self.default_dir.mkdir(parents=True, exist_ok=True)
            if config.export_code:
                self.code_dir.mkdir(parents=True, exist_ok=True)

    def _sanitize_filename(self, filename: str) -> str:
        invalid_chars = '<>:"/\\\\|?*'
        for char in invalid_chars:
            filename = filename.replace(char, '_')
        filename = re.sub(r'[\\x00-\\x1f\\x7f-\\x9f]', '', filename)
        filename = ' '.join(filename.split())
        return filename[:self.config.max_filename_len]

    def _generate_unique_filename(self, base_name: str, directory: Path, ext: str) -> Path:
        filepath = directory / f"{base_name}.{ext}"

        if filepath not in self.filename_cache:
            self.filename_cache[filepath] = True
            return filepath

        for i in range(1, 1000):
            filepath = directory / f"{base_name}_{i:03d}.{ext}"
            if filepath not in self.filename_cache:
                self.filename_cache[filepath] = True
                return filepath

        hash_suffix = hashlib.md5(base_name.encode()).hexdigest()[:6]
        filepath = directory / f"{base_name}_{hash_suffix}.{ext}"
        self.filename_cache[filepath] = True
        return filepath

    def export_conversation(self, conversation: Conversation, target_dir: Optional[Path] = None) -> Optional[Path]:
        if self.config.dry_run:
            return None

        safe_title = self._sanitize_filename(conversation.title)
        date_str = ""
        if conversation.created_at:
            try:
                dt = datetime.fromisoformat(conversation.created_at.replace('Z', '+00:00'))
                date_str = f"_{dt.strftime('%Y%m%d')}"
            except:
                pass

        platform_str = conversation.platform.value
        subset_str = self.config.subset if self.config.subset != 'both' else 'full'
        base_name = f"{platform_str}_{safe_title}{date_str}_{subset_str}"
        ext = self.config.format.value

        if target_dir is None:
            target_dir = self.default_dir

        filepath = self._generate_unique_filename(base_name, target_dir, ext)

        if self.config.format == ExportFormat.MARKDOWN:
            content = ExportFormatter.format_markdown(conversation, self.config.subset)
        elif self.config.format == ExportFormat.TEXT:
            content = ExportFormatter.format_text(conversation, self.config.subset)
        elif self.config.format == ExportFormat.JSON:
            content = ExportFormatter.format_json(conversation, self.config.subset)
        else:
            content = ExportFormatter.format_text(conversation, self.config.subset)

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)

        self.exported_files.append(filepath)
        self.conversation_export_map[conversation.id] = filepath
        self.stats.exported_conversations += 1

        if self.config.verbose:
            logger.info(f"Exported: {filepath.name}")

        return filepath

    def export_code_blocks(self, conversation: Conversation) -> List[Path]:
        if self.config.dry_run or not self.config.export_code:
            return []

        exported_code_files = []
        safe_title = self._sanitize_filename(conversation.title)

        code_num = 1
        for msg in conversation.messages:
            if msg.role == 'assistant':
                code_blocks = CodeExtractor.extract_code_blocks(msg.content)

                for code, language in code_blocks:
                    ext = CodeExtractor.LANGUAGE_EXTENSIONS.get(language, 'txt')
                    base_name = f"{safe_title}_code{code_num:03d}"

                    filepath = self._generate_unique_filename(base_name, self.code_dir, ext.lstrip('.'))

                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(code)

                    exported_code_files.append(filepath)
                    code_num += 1

        return exported_code_files

class MonolithExporter:
    """Export conversations to monolith files with provenance."""

    def __init__(self, config: ExportConfig, exporter: ChatExporter, processor: StreamProcessor):
        self.config = config
        self.exporter = exporter
        self.processor = processor
        self.monolith_dir = config.output_path / "monoliths"

        if not config.dry_run and config.make_monolith:
            self.monolith_dir.mkdir(parents=True, exist_ok=True)

    def _create_index_table(self, conversations: List[Conversation]) -> str:
        """Create an index table for the monolith."""
        lines = ["# Index\n\n"]
        lines.append("| # | Title | Platform | ID | Created | Messages | Export File | Source Files |\n"]
        lines.append("|---|-------|----------|----|---------|---------:|-------------|--------------|\n")

        for i, conv in enumerate(conversations, 1):
            anchor = f"[{conv.title[:50]}...](#{self._make_anchor(conv)})" if len(conv.title) > 50 else f"[{conv.title}](#{self._make_anchor(conv)})"
            export_file = self.exporter.conversation_export_map.get(conv.id, 'N/A')
            if isinstance(export_file, Path):
                export_file = export_file.relative_to(self.config.output_path)

            source_files = ', '.join(self.processor.source_files.get(conv.id, ['Unknown']))

            created = conv.created_at[:10] if conv.created_at else 'N/A'

            lines.append(f"| {i} | {anchor} | {conv.platform.value} | {conv.id[:8]}... | {created} | {conv.message_count} | {export_file} | {source_files} |\n")

        lines.append("\n---\n\n")
        return ''.join(lines)

    def _make_anchor(self, conversation: Conversation) -> str:
        """Create a URL-safe anchor for a conversation."""
        safe_title = re.sub(r'[^\\w\\s-]', '', conversation.title.lower())
        safe_title = re.sub(r'[-\\s]+', '-', safe_title)
        return f"{safe_title}-{conversation.id[:8]}"

    def _format_conversation_section(self, conversation: Conversation) -> str:
        """Format a single conversation section for the monolith."""
        lines = []
        anchor = self._make_anchor(conversation)

        # Header with anchor
        lines.append(f"<a name=\"{anchor}\"></a>\n\n")
        lines.append(f"## {conversation.title}\n\n")

        # Provenance block
        lines.append("### Provenance\n\n")
        export_file = self.exporter.conversation_export_map.get(conversation.id, 'N/A')
        if isinstance(export_file, Path):
            export_file = export_file.relative_to(self.config.output_path)
        source_files = ', '.join(self.processor.source_files.get(conversation.id, ['Unknown']))

        lines.append(f"- **Platform**: {conversation.platform.value}\n")
        lines.append(f"- **Conversation ID**: {conversation.id}\n")
        lines.append(f"- **Created**: {conversation.created_at or 'N/A'}\n")
        lines.append(f"- **Updated**: {conversation.updated_at or 'N/A'}\n")
        lines.append(f"- **Messages**: {conversation.message_count}\n")
        lines.append(f"- **Export File**: `{export_file}`\n")
        lines.append(f"- **Source Files**: {source_files}\n\n")

        # Conversation content
        lines.append("### Content\n\n")

        for msg in conversation.messages:
            if self.config.subset == 'both' or self.config.subset == msg.role:
                role_emoji = "👤" if msg.role == 'user' else "🤖"
                lines.append(f"#### {role_emoji} {msg.role.title()}\n\n")
                lines.append(f"{msg.content}\n\n")
                if msg.timestamp:
                    lines.append(f"*{msg.timestamp}*\n\n")
                lines.append("---\n\n")

        lines.append("\n\n")
        return ''.join(lines)

    def export_monolith(self, conversations: List[Conversation], filename_suffix: str = "") -> Optional[Path]:
        """Export a monolith file containing multiple conversations."""
        if self.config.dry_run or not self.config.make_monolith:
            return None

        # Deduplicate conversations
        seen_keys = set()
        unique_conversations = []

        for conv in conversations:
            key = getattr(conv, self.config.monolith_dedup_key)
            if not key and self.config.monolith_dedup_key == 'id':
                key = conv.unique_hash
            elif not key:
                key = conv.id

            if key not in seen_keys:
                seen_keys.add(key)
                unique_conversations.append(conv)

        # Sort conversations
        unique_conversations.sort(key=lambda c: (c.created_at or '', c.title))

        # Generate filename
        filename = f"{self.config.monolith_filename_prefix}{filename_suffix}.{self.config.monolith_ext.value}"
        filepath = self.monolith_dir / filename

        # Generate content
        content = []

        # Title
        content.append(f"# {self.config.monolith_filename_prefix.title()} Export{filename_suffix}\n\n")
        content.append(f"Generated: {datetime.now().isoformat()}\n\n")
        content.append(f"Total Conversations: {len(unique_conversations)}\n\n")
        content.append("---\n\n")

        # Index
        content.append(self._create_index_table(unique_conversations))

        # Conversations
        content.append("# Conversations\n\n")
        for conv in unique_conversations:
            content.append(self._format_conversation_section(conv))

        # Write file
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(''.join(content))

        logger.info(f"Created monolith: {filepath.name} ({len(unique_conversations)} conversations)")
        return filepath

    def export_all_monoliths(self, all_conversations: List[Conversation],
                           keyword_groups: Dict[str, List[Conversation]] = None):
        """Export monoliths based on configuration."""
        if not self.config.make_monolith:
            return

        exported = []

        if self.config.monolith_mode == 'all':
            # Single monolith with all conversations
            filepath = self.export_monolith(all_conversations)
            if filepath:
                exported.append(filepath)

        elif self.config.monolith_mode == 'by_keyword' and keyword_groups:
            # One monolith per keyword
            for keyword, conversations in keyword_groups.items():
                filepath = self.export_monolith(conversations, f"_{keyword}")
                if filepath:
                    exported.append(filepath)

        if exported:
            print(f"\\n📚 Created {len(exported)} monolith file(s)")

print("✅ Export system defined")

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (ipython-input-1101839506.py, line 218)

In [7]:
# Interactive UI

class InteractiveUI:
    def __init__(self):
        self.config = None
        self.uploaded_files = []
        self.processing = False
        self.conversations = []
        self.stats = ProcessingStats()
        self.setup_ui()

    def setup_ui(self):
        style = {'description_width': 'initial'}
        layout = widgets.Layout(width='100%')

        self.upload_widget = widgets.FileUpload(
            accept='.json,.jsonl,.gz,.zip',
            multiple=True,
            description='Upload JSON Files:',
            style=style
        )

        self.subset_dropdown = widgets.Dropdown(
            options=['both', 'user', 'assistant'],
            value='both',
            description='Message Subset:',
            style=style
        )

        self.format_dropdown = widgets.Dropdown(
            options=['markdown', 'text', 'json'],
            value='markdown',
            description='Export Format:',
            style=style
        )

        self.export_code_checkbox = widgets.Checkbox(
            value=True,
            description='Export Code Blocks',
            style=style
        )

        self.deduplicate_checkbox = widgets.Checkbox(
            value=True,
            description='Remove Duplicates',
            style=style
        )

        self.keywords_text = widgets.Textarea(
            value='',
            placeholder='Enter keywords separated by commas',
            description='Filter Keywords:',
            layout=widgets.Layout(width='100%', height='60px'),
            style=style
        )

        self.filename_length_slider = widgets.IntSlider(
            value=160,
            min=50,
            max=255,
            step=5,
            description='Max Filename Length:',
            style=style,
            layout=layout
        )

        self.process_button = widgets.Button(
            description='🚀 Process Files',
            button_style='success',
            layout=widgets.Layout(width='200px', height='40px')
        )
        self.process_button.on_click(self.on_process_click)

        self.progress_bar = widgets.IntProgress(
            value=0,
            min=0,
            max=100,
            description='Progress:',
            bar_style='info',
            layout=widgets.Layout(width='100%')
        )

        self.output_area = widgets.Output(
            layout=widgets.Layout(width='100%', height='300px', border='1px solid #ddd')
        )

        self.stats_html = widgets.HTML(
            value='<p>No statistics yet. Process files to see results.</p>'
        )

        basic_tab = widgets.VBox([
            widgets.HTML('<h3>📁 Files</h3>'),
            self.upload_widget,
            widgets.HTML('<h3>⚙️ Settings</h3>'),
            self.subset_dropdown,
            self.format_dropdown,
            self.export_code_checkbox,
            self.deduplicate_checkbox
        ])

        advanced_tab = widgets.VBox([
            widgets.HTML('<h3>🔍 Filtering</h3>'),
            self.keywords_text,
            self.filename_length_slider
        ])

        self.tabs = widgets.Tab([basic_tab, advanced_tab])
        self.tabs.set_title(0, '📋 Basic')
        self.tabs.set_title(1, '🔧 Advanced')

        self.main_container = widgets.VBox([
            self.tabs,
            widgets.HBox([self.process_button]),
            self.progress_bar,
            self.stats_html,
            self.output_area
        ])

    def on_process_click(self, button):
        with self.output_area:
            clear_output()
            print("🚀 Starting processing...")

            try:
                self.create_config()
                errors = self.config.validate()
                if errors:
                    print("❌ Configuration errors:")
                    for error in errors:
                        print(f"  - {error}")
                    return

                self.process_files()
                self.update_statistics()
                print("\\n✅ Processing complete!")

            except Exception as e:
                print(f"\\n❌ Error: {str(e)}")
                traceback.print_exc()

    def create_config(self):
        keywords = [k.strip() for k in self.keywords_text.value.split(',') if k.strip()]

        format_map = {
            'markdown': ExportFormat.MARKDOWN,
            'text': ExportFormat.TEXT,
            'json': ExportFormat.JSON
        }

        self.config = ExportConfig(
            input_path=Path('/content/input_data'),
            output_path=Path('/content/chat_exports'),
            subset=self.subset_dropdown.value,
            format=format_map[self.format_dropdown.value],
            export_code=self.export_code_checkbox.value,
            keywords=keywords,
            max_filename_len=self.filename_length_slider.value,
            deduplicate=self.deduplicate_checkbox.value,
            verbose=True
        )

        if self.upload_widget.value:
            input_dir = Path('/content/input_data')
            input_dir.mkdir(exist_ok=True)

            for file_info in self.upload_widget.value.values():
                file_path = input_dir / file_info['metadata']['name']
                with open(file_path, 'wb') as f:
                    f.write(file_info['content'])
                print(f"📥 Saved: {file_info['metadata']['name']}")

    def process_files(self):
        parser = UniversalChatParser()
        processor = StreamProcessor(parser)
        exporter = ChatExporter(self.config)

        print("\\n📂 Processing files...")
        self.conversations = processor.process_directory(self.config.input_path, self.config)

        print("\\n🔍 Applying filters...")
        self.conversations = ConversationFilter.apply_filters(self.conversations, self.config)

        print(f"\\n📤 Exporting {len(self.conversations)} conversations...")
        self.progress_bar.max = len(self.conversations)

        for i, conv in enumerate(self.conversations):
            exporter.export_conversation(conv)
            if self.config.export_code:
                exporter.export_code_blocks(conv)
            self.progress_bar.value = i + 1

        self.stats = processor.stats
        self.stats.exported_conversations = len(self.conversations)

    def update_statistics(self):
        platform_stats = ""
        for platform, count in self.stats.platform_breakdown.items():
            platform_stats += f"<li>{platform.value}: {count}</li>"

        html_content = f"""
        <div style="background: #f5f5f5; padding: 15px; border-radius: 10px;">
            <h3>📊 Statistics</h3>
            <p>Files: {self.stats.processed_files}/{self.stats.total_files}</p>
            <p>Conversations: {self.stats.total_conversations}</p>
            <p>Exported: {self.stats.exported_conversations}</p>
            <ul>{platform_stats}</ul>
        </div>
        """
        self.stats_html.value = html_content

    def display(self):
        display(self.main_container)

# Create and display UI
ui = InteractiveUI()
ui.display()
print("\\n💡 Use the interface above to configure and run your export!")

\n💡 Use the interface above to configure and run your export!


In [ ]:
# Download Results

def create_download_archive():
    output_root = Path('/content/chat_exports')

    if not output_root.exists():
        print("❌ No exports found. Please process files first.")
        return

    all_files = list(output_root.rglob('*'))
    file_count = sum(1 for f in all_files if f.is_file())

    if file_count == 0:
        print("❌ No files to archive.")
        return

    print(f"📦 Creating archive of {file_count} files...")

    zip_path = Path('/content/chat_exports.zip')

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in tqdm(all_files, desc="Archiving"):
            if file_path.is_file():
                arcname = file_path.relative_to(output_root.parent)
                zipf.write(file_path, arcname)

    file_size_mb = zip_path.stat().st_size / (1024**2)
    print(f"\\n✅ Archive created: {zip_path}")
    print(f"📊 Size: {file_size_mb:.2f} MB")
    print(f"📁 Files: {file_count}")

    from google.colab import files
    files.download(str(zip_path))

create_download_archive()

---

## 📚 Documentation

### Features
- Multi-platform support (ChatGPT, Claude, Gemini)
- Memory-efficient streaming for large files
- Interactive UI with ipywidgets
- User message cleaning with three independent filters
- Monolith export with full provenance tracking
- Multiple export formats
- Code block extraction
- Advanced filtering options

### User Message Cleaning
**Three Independent Filters:**
1. **Length Gate**: Drop messages shorter than threshold
2. **Exact Dedup**: Remove exact duplicates (per conversation or global)
3. **Near Dedup**: Remove similar messages (Jaccard similarity)

**Presets:**
- **None**: All filters disabled
- **Light**: Length (120 chars) + exact dedup
- **Heavy**: Length (160 chars) + exact dedup + near dedup (0.90 threshold)

### Monolith Export
Creates merged files containing multiple conversations with:
- Index table with metadata
- Full provenance (source files, export paths)
- Sorted by creation date
- Deduplication by ID or hash

### Troubleshooting
- Ensure JSON files are valid format
- Use stream mode for large files (>100MB)
- Check `/content/input_data/` for uploaded files

### Version History
- **v2.1.0** (Current): Added user message cleaning and monolith export
- **v2.0.0**: Complete rewrite with streaming and UI
- **v1.0.0**: Initial version with basic export functionality

---
Created with ❤️ for the AI community